In [1]:
script = '''from ultralytics import YOLO
import pandas as pd
import yaml
import os

DSS = "/dss/dssfs04/lwp-dss-0002/pn67pu/pn67pu-dss-0000/ra26fuz"

SUMMER_WEIGHTS = f"{DSS}/results/final_summer_yolo_v6/final/weights/best.pt"
FALL_WEIGHTS   = f"{DSS}/results/final_fall_yolo_v6/final/weights/best.pt"
SUMMER_TEST    = f"{DSS}/data/test_2cls/test/images"
FALL_TEST      = f"{DSS}/data/test_fall_2cls/test/images"
OUTPUT_DIR     = f"{DSS}/results/test_set_evaluation_yolo_v6"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_yaml(images_path, output_path):
    content = {
        "path": "/",
        "train": images_path,
        "val":   images_path,
        "test":  images_path,
        "nc": 2,
        "names": {0: "deciduous_tree", 1: "shrub"}
    }
    with open(output_path, "w") as f:
        yaml.dump(content, f, default_flow_style=False)
    print(f"Created yaml: {output_path}")

summer_yaml = f"{OUTPUT_DIR}/test_summer.yaml"
fall_yaml   = f"{OUTPUT_DIR}/test_fall.yaml"
create_yaml(SUMMER_TEST, summer_yaml)
create_yaml(FALL_TEST,   fall_yaml)

def evaluate_model(weights_path, data_yaml, season):
    print(f"\\n{\'=\'*50}")
    print(f"Evaluating {season} YOLOv8m v6 on test set...")
    model = YOLO(weights_path)
    metrics = model.val(
        data     = data_yaml,
        imgsz    = 640,
        batch    = 16,
        device   = 0,
        save_txt = True,
        save_conf= True,
        project  = OUTPUT_DIR,
        name     = f"test_{season}",
        exist_ok = True,
        plots    = True,
        conf     = 0.001,
        iou      = 0.5,
    )
    return {
        "mAP50":          round(metrics.box.map50, 4),
        "F1":             round(metrics.box.f1.mean(), 4),
        "precision":      round(metrics.box.p.mean(), 4),
        "recall":         round(metrics.box.r.mean(), 4),
        "mAP50_95":       round(metrics.box.map, 4),
        "AP50_deciduous": round(metrics.box.ap50[0], 4),
        "AP50_shrub":     round(metrics.box.ap50[1], 4),
    }

summer_res = evaluate_model(SUMMER_WEIGHTS, summer_yaml, "summer")
fall_res   = evaluate_model(FALL_WEIGHTS,   fall_yaml,   "fall")

print(f"\\n=== SUMMER YOLO v6 TEST SET ===")
for k, v in summer_res.items(): print(f"  {k}: {v}")
print(f"\\n=== FALL YOLO v6 TEST SET ===")
for k, v in fall_res.items(): print(f"  {k}: {v}")

pd.DataFrame([
    {**summer_res, "model": "YOLOv8m v6 final", "season": "summer",
     "dataset": "59 test tiles", "n_tiles": 59},
    {**fall_res,   "model": "YOLOv8m v6 final", "season": "fall",
     "dataset": "59 test tiles", "n_tiles": 59},
]).to_csv(f"{OUTPUT_DIR}/yolo_v6_test_results.csv", index=False)
print(f"\\nSaved: {OUTPUT_DIR}/yolo_v6_test_results.csv")
'''

path = r"C:\Master Physische Geographie\Canada\Masterarbeit\vscode\evaluate_testset_yolo_v6.py"
with open(path, "w") as f:
    f.write(script)
print(f"Saved locally: {path}")

Saved locally: C:\Master Physische Geographie\Canada\Masterarbeit\vscode\evaluate_testset_yolo_v6.py
